# Underwrite: source audit and exploratory analysis
Run from project root. The class label is encoded in a derived Figshare snapshot; validate provenance before interpreting outcomes. All plots use only training-period observations.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from backend.ml.features import FEATURES, clean_frame
path = Path('data/raw/train_lending_club.csv')
raw = pd.read_csv(path)
print(raw.shape, raw.issue_d.min(), raw.issue_d.max())
raw.head()

## Data quality and target audit
The file contains historical funded loans, so `loan_status` is an outcome label rather than approval. Compare label-conditioned FICO, interest rate and DTI before treating 1 as favorable repayment.

In [ ]:
print(raw.loan_status.value_counts(dropna=False))
print(raw.groupby('loan_status')[['fico_range_low','int_rate','dti','loan_amnt']].mean().round(2))
print(raw[FEATURES].isna().mean().sort_values(ascending=False))

## Chronological partitions
Fit on 2014–2015; calibrate on Jan–Mar 2016; final test on Apr–Jun 2016. EDA below avoids test-set tuning.

In [ ]:
x = clean_frame(raw)
x.issue_d = pd.to_datetime(raw.issue_d)
fit = x[x.issue_d < '2016-01-01']
cal = x[(x.issue_d >= '2016-01-01') & (x.issue_d < '2016-04-01')]
test = x[x.issue_d >= '2016-04-01']
print({'fit':len(fit),'calibration':len(cal),'test':len(test)})

## Training-period distributions
Check outliers, cohort drift and correlations. This is descriptive, not a causal fairness analysis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
fit.loan_amnt.hist(ax=axes[0], bins=35); axes[0].set_title('Historical funded amount')
fit.fico_range_low.hist(ax=axes[1], bins=30); axes[1].set_title('FICO lower bound')
fit.groupby('issue_d').loan_status.mean().plot(ax=axes[2]); axes[2].set_title('Observed favorable label by issue month')
plt.tight_layout()

## Reproduce models
Training is modularized in `backend/ml/train.py`, keeping notebook experimentation separate from serving. Inspect `models/metrics.json` after the run. Historical amount is a benchmark, not an optimal offer.

In [ ]:
from backend.ml.train import train
train(path, Path('models'))
import json
print(json.dumps(json.loads(Path('models/metrics.json').read_text()), indent=2))